# Om den här mallen {.unnumbered}

Det här är **webbversionen** av JTH:s labbrapportmall. Den ger en enda
HTML-fil som du kan mejla eller lämna in, och som kan innehålla sådant
papper inte klarar: animationer, filmer, ljud och grafer man kan zooma i.

::: {.callout-tip}
## Vilken version ska jag välja?
| | `report.ipynb` (PDF) | `report-html.ipynb` (den här) |
|---|---|---|
| Inlämning på papper, tryck | ja | nej |
| Fasta figurer, ekvationer, referenser | ja | ja |
| Animationer, film, ljud | nej | ja |
| Interaktiva grafer (zooma, hovra) | nej | ja |
| Kod som läsaren kan fälla ut | nej | ja |

Följ alltid kursens instruktion. Är inget sagt: lämna in PDF.
:::

Rendera med knappen **Render to html using Quarto** i verktygsraden, eller
**Render & preview** för att se resultatet direkt. Tack vare
`embed-resources: true` blir allt en enda fil — inga bilder eller filmer
som kan glömmas kvar.

Radera det här avsnittet när du skriver din egen rapport.

# Inledning

Beskriv uppgiften, syftet och vad läsaren ska få ut av rapporten. Referera
till källor med hakparenteser, som `[@Stromberg2012]` → [@Stromberg2012].

Matematik skrivs som i PDF-versionen. I en mening: $F = ma$. Fristående, med
en etikett som kan refereras:

$$
\int_0^1 x^2\,dx = \frac{1}{3}
$$ {#eq-integral}

Se @eq-integral.

# Teori

Kort, men tillräckligt för att läsaren ska förstå beräkningarna. Använd
gärna en faktaruta:

::: {.callout-note}
## Antagande
Luftmotståndet försummas i hela rapporten.
:::

# Metod och resultat

## Figurer från kod

Precis som i PDF-versionen: ge cellen en etikett och en figurtext, så
numreras figuren och kan refereras med `@fig-sinus`.

In [ ]:
#| label: fig-sinus
#| fig-cap: "En sinusvåg genererad med Python"
#| code-fold: show
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 4 * np.pi, 400)
plt.figure(figsize=(6, 3))
plt.plot(t, np.sin(t))
plt.xlabel("t [s]")
plt.ylabel("u(t) [V]")
plt.grid(alpha=.3)
plt.show()

Se @fig-sinus. Med `code-fold: show` syns koden med en pil för att fälla
in den; `#| code-fold: true` börjar infälld och `#| echo: false` gömmer den helt.

## Tabeller

En pandas-tabell blir en HTML-tabell. `#| tbl-cap` ger tabelltext och numrering.

In [ ]:
#| label: tbl-matning
#| tbl-cap: "Uppmätta värden för tre försök"
import pandas as pd

df = pd.DataFrame({
    "Försök": [1, 2, 3],
    "Massa m [kg]": [0.50, 0.75, 1.00],
    "Kraft F [N]": [4.91, 7.36, 9.81],
})
df["a = F/m [m/s²]"] = (df["Kraft F [N]"] / df["Massa m [kg]"]).round(2)
df

Se @tbl-matning.

# Rörliga bilder

Tre sätt att få en animation in i rapporten. Alla tre bygger på
`matplotlib.animation.FuncAnimation`; skillnaden är hur den sparas.
Exemplen följer kursmaterialet på
[python.ju.se](https://python.ju.se/ProgrammingFundamentals/RichMediaReports.html).

## Sätt 1: GIF — enklast

Fungerar överallt, men filen blir stor och färgerna begränsade (256 färger).
Bra för korta, enkla förlopp.

In [ ]:
#| label: gif-animation
#| echo: true
#| output: false
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(4, 4))
line, = ax.plot([], [], "ro")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
t = np.linspace(0, 2 * np.pi, 60)
ax.plot(np.cos(t), np.sin(t), "b-")
ax.set_aspect("equal", "box")


def update(frame):
    line.set_data([np.cos(frame)], [np.sin(frame)])
    return line,


ani = animation.FuncAnimation(fig, update, frames=t, blit=True)
plt.close()
ani.save("circle_animation.gif", writer="pillow", fps=20, dpi=80)

![Punkt som går runt enhetscirkeln (GIF)](circle_animation.gif){width=60%}

## Sätt 2: MP4 — bäst kvalitet

`ffmpeg` finns på servern. Filmen blir liten och skarp, och läsaren får
kontroller för att spela, pausa och spola.

In [ ]:
#| label: mp4-animation
#| echo: true
#| output: false
fig, ax = plt.subplots(figsize=(4, 4))
line, = ax.plot([], [], "ro")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
t = np.linspace(0, 2 * np.pi, 240)
ax.plot(np.cos(t), np.sin(t), "b-")
ax.set_aspect("equal", "box")
ani = animation.FuncAnimation(fig, update, frames=t, blit=True)
plt.close()
ani.save("circle_animation.mp4", writer="ffmpeg", fps=30, dpi=150)

<video width="60%" controls autoplay loop muted>
  <source src="circle_animation.mp4" type="video/mp4">
  Din läsare kan inte visa filmen.
</video>

## Sätt 3: Interaktiv uppspelning

`to_jshtml` bygger en spelare med bildruteväljare, så läsaren kan stega fram
och tillbaka. Notera `#| output: asis` — HTML-koden ska in som den är.

In [ ]:
#| label: jshtml-animation
#| output: asis
#| code-fold: true
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(3.2, 3.2))
line, = ax.plot([], [], "ro")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
t = np.linspace(0, 2 * np.pi, 30)
ax.plot(np.cos(t), np.sin(t), "b-")
ax.set_aspect("equal", "box")
ani = animation.FuncAnimation(fig, update, frames=t, blit=True)
plt.close()

HTML(f'''<div style="width:100%">
<style>img {{ max-width: 100% !important; }}</style>
{ani.to_jshtml(default_mode="loop")}
</div>''')

::: {.callout-warning}
## Storlek
En animation kan väga några MB, och med `embed-resources: true` hamnar allt
i HTML-filen. Håll animationerna korta (ett par sekunder) och `dpi` måttlig,
annars blir filen tung att mejla.
:::

# Interaktiva grafer

Med `plotly` kan läsaren zooma, panorera och läsa av värden genom att peka.
Bra när mätserien är tät eller när detaljer göms i en statisk bild.

In [ ]:
#| label: fig-plotly
#| fig-cap: "Dämpad svängning — peka i grafen för att läsa av värden"
#| code-fold: true
#| output: asis
import numpy as np
import plotly.graph_objects as go

t = np.linspace(0, 10, 500)
x = np.exp(-0.3 * t) * np.cos(2 * np.pi * t)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=x, mode="lines", name="x(t)"))
fig.add_trace(go.Scatter(x=t, y=np.exp(-0.3 * t), mode="lines",
                         name="hölje", line=dict(dash="dash")))
fig.update_layout(template="simple_white", height=380,
                  xaxis_title="t [s]", yaxis_title="x(t) [m]",
                  margin=dict(l=60, r=20, t=30, b=50))

# include_plotlyjs="inline" bäddar in ritbiblioteket, så grafen fungerar
# utan internet. Byt till "cdn" om du vill ha en liten fil.
from IPython.display import HTML
HTML(fig.to_html(full_html=False, include_plotlyjs="inline"))

Se @fig-plotly.

::: {.callout-note collapse="true"}
## Plotly gör filen ~5 MB större
Ritbiblioteket bäddas in (`include_plotlyjs="inline"`) så att grafen fungerar
utan internet. Vill du ha en liten fil i stället, och kan räkna med att
läsaren är uppkopplad, byt sista raden i cellen mot:

```python
HTML(fig.to_html(full_html=False, include_plotlyjs="cdn"))
```

Behöver du ingen interaktivitet räcker matplotlib, och filen blir bara några
hundra kB.
:::

# Ljud

Ett ljud säger mer än en spektrumbild när det gäller till exempel svävningar
eller vibrationer.

In [ ]:
#| label: ljud
#| code-fold: true
import numpy as np
from IPython.display import Audio

fs = 22050
t = np.linspace(0, 2, 2 * fs, endpoint=False)
# två närliggande frekvenser ger svävning
signal = 0.4 * (np.sin(2 * np.pi * 220 * t) + np.sin(2 * np.pi * 223 * t))
Audio(signal, rate=fs)

# Flikar och annat som bara finns på webben

::: {.panel-tabset}
## Resultat
Kort sammanfattning för den som bara vill se svaret.

## Beräkning
$$ a = \frac{F}{m} = \frac{9{,}81}{1{,}00} = 9{,}81\ \mathrm{m/s^2} $$

## Kod
```python
a = F / m
```
:::

::: {.callout-note collapse="true"}
## Flödesscheman (lägger till ~3 MB)
Quarto kan rita diagram direkt i texten med ```` ```{mermaid} ````. Det är
snyggt, men ritbiblioteket bäddas in och filen växer med ungefär 3 MB, så
mallen använder det inte som standard.
:::

# Diskussion och slutsats

Tolka resultaten, jämför med teorin och var tydlig med felkällor.

# Så delar du rapporten

Renderingen ger **en** fil, `report-html.html`, med allt inbakat:

* ladda ner den (högerklick på filen i filbläddraren → Download) och mejla in den, eller
* lämna in den där kursen säger.

Mallen som den står ger ungefär 11 MB, varav plotly står för hälften. Blir
filen för tung att mejla:

* ta bort avsnitt du inte använder (plotly väger ~5 MB, animationerna ~1 MB),
* korta animationerna och sänk `dpi`,
* eller sätt `embed-resources: false` — då hamnar filmer och bilder i mappen
  `report-html_files/`, som måste följa med filen.